In [ ]:
# ==============================================================
#   OASIS (CNN)  vs  CCNA (Tabular)  —  Behavioral Similarity
#   Metrics: JSD (→ similarity as 1-JSD), Bhattacharyya, Hellinger, R_prob
# ==============================================================

import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon
import math
import matplotlib.pyplot as plt

# -------------------------
# Config
# -------------------------
CCNA_PATH = "ccna_pred_probs.csv"
OASIS_PATH = "image_preds_with_id.csv"
PAIR_BY_ID = False  # set True only if you later create an ID-aligned file

# -------------------------
# Helpers
# -------------------------
def softmax_onehot(labels, n_classes):
    """Turn integer class labels into one-hot probabilities."""
    arr = np.zeros((len(labels), n_classes), dtype=float)
    for i, c in enumerate(labels):
        c = int(c)
        if 0 <= c < n_classes:
            arr[i, c] = 1.0
    # tiny smoothing to avoid zeros in divergences
    eps = 1e-9
    arr = (arr + eps) / (arr.sum(axis=1, keepdims=True) + n_classes*eps)
    return arr

def normalize_rows(p):
    """Ensure each row is a valid probability simplex."""
    p = np.clip(p, 1e-12, None)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def bc_coeff(p, q):
    """Bhattacharyya coefficient for probability vectors (row-wise)."""
    return np.sum(np.sqrt(p * q), axis=-1)

def hellinger_from_bc(bc):
    return np.sqrt(1.0 - np.clip(bc, 0.0, 1.0))

def jsd_similarity(p, q):
    """Return similarity = 1 - JSD^2 (SciPy jensenshannon returns sqrt(JS divergence))."""
    js = jensenshannon(p, q)  # this is sqrt(JS)
    return 1.0 - float(js**2)

# -------------------------
# Load CCNA (tabular) — must contain ccna_prob_*
# -------------------------
ccna = pd.read_csv(CCNA_PATH)
prob_cols_ccna = [c for c in ccna.columns if c.startswith("ccna_prob_")]
if not prob_cols_ccna:
    raise ValueError("No 'ccna_prob_*' columns found in CCNA file.")
P_ccna = ccna[prob_cols_ccna].values.astype(float)
P_ccna = normalize_rows(P_ccna)

# infer CCNA class ids from column suffixes
ccna_classes = [int(c.split("_")[-1]) for c in prob_cols_ccna]
true_ccna = ccna["true_label"].to_numpy() if "true_label" in ccna.columns else None

# -------------------------
# Load OASIS (CNN) — may only have predicted labels
# -------------------------
oasis = pd.read_csv(OASIS_PATH)

# Try to detect per-class probs for OASIS; otherwise build 1-hot from 'image_pred'
prob_cols_oasis = [c for c in oasis.columns if c.startswith("oasis_prob_")]
if prob_cols_oasis:
    P_oasis = oasis[prob_cols_oasis].values.astype(float)
    P_oasis = normalize_rows(P_oasis)
    oasis_classes = [int(c.split("_")[-1]) for c in prob_cols_oasis]
else:
    if "image_pred" not in oasis.columns:
        raise ValueError("OASIS file missing per-class probs and 'image_pred'. Add one of them.")
    # Build one-hot probs using the **same label space as CCNA** by default
    n_classes = len(ccna_classes)
    P_oasis = softmax_onehot(oasis["image_pred"].to_numpy(), n_classes)
    oasis_classes = ccna_classes.copy()

true_oasis = oasis["true_label"].to_numpy() if "true_label" in oasis.columns else None

# -------------------------
# Align on overlapping class ids
# -------------------------
overlap_classes = sorted(set(ccna_classes).intersection(set(oasis_classes)))
if len(overlap_classes) < 2:
    raise ValueError(f"Not enough overlapping classes to compare. Overlap={overlap_classes}")

# Reindex probabilities to the overlap in consistent order
def reindex_probs(P, src_classes, target_classes):
    idx = [src_classes.index(k) for k in target_classes]
    return P[:, idx]

P_ccna_ov = reindex_probs(P_ccna, ccna_classes, overlap_classes)
P_oasis_ov = reindex_probs(P_oasis, oasis_classes, overlap_classes)

# Also filter true labels if available
def filter_true(y, keep_classes):
    if y is None:
        return None
    mask = np.isin(y, keep_classes)
    return y[mask], mask

true_ccna_ov, mask_ccna = filter_true(true_ccna, overlap_classes)
true_oasis_ov, mask_oasis = filter_true(true_oasis, overlap_classes)

if mask_ccna is not None:
    P_ccna_ov = P_ccna_ov[mask_ccna]
if mask_oasis is not None:
    P_oasis_ov = P_oasis_ov[mask_oasis]

# -------------------------
# Comparison mode A: Aggregate (class-wise)
# -------------------------
# For each class k in the overlap, compute the mean probability vector among samples of class k
def classwise_means(P, y, classes):
    means = []
    counts = []
    if y is None:
        # If we don't have true labels, just use global mean once
        return [P.mean(axis=0)], [len(P)]
    for k in classes:
        sel = (y == k)
        if sel.sum() == 0:
            # put a tiny uniform vector if class absent
            means.append(np.ones(P.shape[1]) / P.shape[1])
            counts.append(0)
        else:
            means.append(P[sel].mean(axis=0))
            counts.append(int(sel.sum()))
    return means, counts

cw_ccna, cnt_ccna = classwise_means(P_ccna_ov, true_ccna_ov, overlap_classes)
cw_oasis, cnt_oasis = classwise_means(P_oasis_ov, true_oasis_ov, overlap_classes)

# Compute metrics per class, then average (weighted by min(count_ccna, count_oasis) to be conservative)
rows = []
weights = []
for k, (p_k, q_k, n_c, n_o) in enumerate(zip(cw_ccna, cw_oasis, cnt_ccna, cnt_oasis)):
    p = np.array(p_k, dtype=float); p = p / p.sum()
    q = np.array(q_k, dtype=float); q = q / q.sum()
    js_sim = jsd_similarity(p, q)
    bc = float(np.sum(np.sqrt(p*q)))
    h = float(math.sqrt(max(0.0, 1.0 - bc)))
    rprob = (bc + js_sim) / 2.0
    rows.append({"class": overlap_classes[k], "JSD_similarity": js_sim, "Bhattacharyya": bc, "Hellinger": h, "R_prob": rprob})
    weights.append(min(n_c, n_o))

# Weighted average across classes (fallback to uniform if all zeros)
w = np.array(weights, dtype=float)
if w.sum() == 0:
    w = np.ones_like(w)
w = w / w.sum()

agg_js = float(np.sum(w * np.array([r["JSD_similarity"] for r in rows])))
agg_bc = float(np.sum(w * np.array([r["Bhattacharyya"] for r in rows])))
agg_h  = float(np.sum(w * np.array([r["Hellinger"] for r in rows])))
agg_rp = float(np.sum(w * np.array([r["R_prob"] for r in rows])))

summary = pd.DataFrame(rows + [
    {"class": "WEIGHTED_MEAN", "JSD_similarity": agg_js, "Bhattacharyya": agg_bc, "Hellinger": agg_h, "R_prob": agg_rp}
])
print("\n📊 BEHAVIORAL SIMILARITY (Class-wise, weighted):")
print(summary)

summary.to_csv("oasis_ccna_similarity_classwise.csv", index=False)

# -------------------------
# Comparison mode B: Global (single vector each)
# -------------------------
p_global = P_ccna_ov.mean(axis=0); p_global /= p_global.sum()
q_global = P_oasis_ov.mean(axis=0); q_global /= q_global.sum()
global_js = jsd_similarity(p_global, q_global)
global_bc = float(np.sum(np.sqrt(p_global*q_global)))
global_h  = float(math.sqrt(max(0.0, 1.0 - global_bc)))
global_rp = (global_bc + global_js) / 2.0

global_df = pd.DataFrame([{
    "scope": "GLOBAL",
    "JSD_similarity": global_js,
    "Bhattacharyya": global_bc,
    "Hellinger": global_h,
    "R_prob": global_rp
}])
print("\n🌐 GLOBAL SIMILARITY (Aggregated over all samples):")
print(global_df)
global_df.to_csv("oasis_ccna_similarity_global.csv", index=False)

# -------------------------
# Optional: Paired mode (only if you later align IDs)
# -------------------------
if PAIR_BY_ID and ("ID" in ccna.columns) and ("ID" in oasis.columns):
    m = pd.merge(
        ccna[["ID"] + prob_cols_ccna],
        oasis[["ID"] + ([f"oasis_prob_{i}" for i in overlap_classes] if prob_cols_oasis else ["image_pred"])],
        on="ID", suffixes=("_ccna", "_oasis")
    )
    # If OASIS has only labels in paired mode, turn into one-hot on overlap
    if not prob_cols_oasis:
        onehot = softmax_onehot(m["image_pred"].to_numpy(), len(overlap_classes))
        m = pd.concat([m.drop(columns=["image_pred"]), pd.DataFrame(onehot, columns=[f"oasis_prob_{i}" for i in overlap_classes])], axis=1)

    Pp_ccna = m[[f"ccna_prob_{i}" for i in overlap_classes]].values
    Pp_oasis = m[[f"oasis_prob_{i}" for i in overlap_classes]].values
    Pp_ccna = normalize_rows(Pp_ccna)
    Pp_oasis = normalize_rows(Pp_oasis)

    js_list, bc_list, h_list, rp_list = [], [], [], []
    for i in range(len(m)):
        p = Pp_ccna[i]; q = Pp_oasis[i]
        js_list.append(jsd_similarity(p, q))
        bc = np.sum(np.sqrt(p*q)); bc_list.append(float(bc))
        h_list.append(float(math.sqrt(max(0.0, 1.0 - bc))))
        rp_list.append((bc_list[-1] + js_list[-1]) / 2.0)

    paired = pd.DataFrame({
        "JSD_similarity": js_list,
        "Bhattacharyya": bc_list,
        "Hellinger": h_list,
        "R_prob": rp_list
    })
    paired.to_csv("oasis_ccna_similarity_paired.csv", index=False)
    print("\n🔗 PAIRED (ID-aligned) similarity summary:")
    print(paired.describe().round(4))

# -------------------------
# Plots (simple & clean)
# -------------------------
# 1) Bar: global mean probabilities per class (both models)
classes_txt = [str(k) for k in overlap_classes]
x = np.arange(len(overlap_classes))
width = 0.35

plt.figure(figsize=(6,4))
plt.bar(x - width/2, p_global, width, label="CCNA")
plt.bar(x + width/2, q_global, width, label="OASIS")
plt.xticks(x, classes_txt)
plt.ylabel("Mean class probability")
plt.title("Global mean predicted probabilities")
plt.legend()
plt.tight_layout()
plt.savefig("global_mean_probs.png", dpi=300)
plt.close()

# 2) Line: per-class R_prob / (1-JSD) / BC (class-wise)
plt.figure(figsize=(6,4))
plt.plot([str(r["class"]) for r in rows], [r["R_prob"] for r in rows], marker="o", label="R_prob")
plt.plot([str(r["class"]) for r in rows], [r["JSD_similarity"] for r in rows], marker="o", label="1 - JSD")
plt.plot([str(r["class"]) for r in rows], [r["Bhattacharyya"] for r in rows], marker="o", label="Bhattacharyya")
plt.ylabel("Score (0–1)")
plt.title("Per-class behavioral similarity")
plt.legend()
plt.tight_layout()
plt.savefig("per_class_similarity.png", dpi=300)
plt.close()

# -------------------------
# LaTeX table (compact)
# -------------------------
def to_latex_table(df, caption, label):
    cols = df.columns.tolist()
    latex = "\\begin{table}[ht]\\centering\n"
    latex += "\\small\n"
    latex += df.to_latex(index=False, float_format=lambda x: f"{x:.3f}")
    latex += f"\\caption{{{caption}}}\n"
    latex += f"\\label{{{label}}}\n"
    latex += "\\end{table}\n"
    return latex

with open("oasis_ccna_similarity_classwise.tex", "w") as f:
    f.write(to_latex_table(summary,
        "Behavioral similarity between OASIS and CCNA models (class-wise, weighted).",
        "tab:oasis-ccna-classwise"))

with open("oasis_ccna_similarity_global.tex", "w") as f:
    f.write(to_latex_table(global_df,
        "Global behavioral similarity between OASIS and CCNA models.",
        "tab:oasis-ccna-global"))

print("\n📝 Saved:")
print(" - oasis_ccna_similarity_classwise.csv / .tex")
print(" - oasis_ccna_similarity_global.csv / .tex")
print(" - global_mean_probs.png")
print(" - per_class_similarity.png")



📊 BEHAVIORAL SIMILARITY (Class-wise, weighted):
           class  JSD_similarity  Bhattacharyya  Hellinger    R_prob
0              0        0.940228       0.935013   0.254925  0.937620
1              1        0.809333       0.739598   0.510296  0.774466
2              2        0.317011       0.054632   0.972301  0.185821
3  WEIGHTED_MEAN        0.870087       0.830948   0.385294  0.850517

🌐 GLOBAL SIMILARITY (Aggregated over all samples):
    scope  JSD_similarity  Bhattacharyya  Hellinger    R_prob
0  GLOBAL        0.913332       0.897615   0.319976  0.905473

📝 Saved:
 - oasis_ccna_similarity_classwise.csv / .tex
 - oasis_ccna_similarity_global.csv / .tex
 - global_mean_probs.png
 - per_class_similarity.png


In [ ]:
# ==============================================================
#    Cross-Cohort Behavioral Similarity (OASIS ↔ CCNA)
#    with Pearson, Kendall’s τ, and Bootstrap Confidence Intervals
# ==============================================================

import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon
from scipy.stats import pearsonr, kendalltau
import math
from tqdm import trange

# -------------------------
# Load inputs
# -------------------------
ccna = pd.read_csv("ccna_pred_probs.csv")
oasis = pd.read_csv("image_preds_with_id.csv")

# detect number of classes
prob_cols_ccna = [c for c in ccna.columns if c.startswith("ccna_prob_")]
n_classes = len(prob_cols_ccna)

# Build OASIS probability matrix (one-hot if needed)
if any(c.startswith("oasis_prob_") for c in oasis.columns):
    prob_cols_oasis = [c for c in oasis.columns if c.startswith("oasis_prob_")]
    P_oasis = oasis[prob_cols_oasis].to_numpy(float)
else:
    assert "image_pred" in oasis.columns
    P_oasis = np.zeros((len(oasis), n_classes))
    for i, label in enumerate(oasis["image_pred"].astype(int)):
        if label < n_classes:
            P_oasis[i, label] = 1.0

P_ccna = ccna[prob_cols_ccna].to_numpy(float)

# -------------------------
# Aggregate mean probability distributions
# -------------------------
p = P_ccna.mean(axis=0); p /= p.sum()
q = P_oasis.mean(axis=0); q /= q.sum()

# -------------------------
# Metrics
# -------------------------
def js_similarity(p, q):
    js = jensenshannon(p, q)
    return 1 - js**2

def bc_coeff(p, q):
    return np.sum(np.sqrt(p * q))

def hellinger_similarity(p, q):
    bc = bc_coeff(p, q)
    return 1 - math.sqrt(1 - bc)

def pearson_similarity(p, q):
    r, _ = pearsonr(p, q)
    return (r + 1) / 2  # map −1→0, +1→1

def kendall_tau_similarity(p, q):
    tau, _ = kendalltau(np.argsort(p), np.argsort(q))
    return tau  # keep as −1 → 1

JS = js_similarity(p, q)
BC = bc_coeff(p, q)
H_sim = hellinger_similarity(p, q)
P_sim = pearson_similarity(p, q)
K_tau = kendall_tau_similarity(p, q)

# -------------------------
# Composite indices
# -------------------------
R_prob = (JS + BC + P_sim + H_sim) / 4
R_all = (R_prob + K_tau) / 2

# -------------------------
# Bootstrap 95 % CI
# -------------------------
boot_Rprob = []
rng = np.random.default_rng(42)
for _ in trange(1000, desc="Bootstrapping"):
    idx = rng.choice(len(P_ccna), len(P_ccna), replace=True)
    pb = P_ccna[idx].mean(axis=0); pb /= pb.sum()
    qb = P_oasis[idx % len(P_oasis)].mean(axis=0); qb /= qb.sum()
    boot_Rprob.append((js_similarity(pb, qb)
                      + bc_coeff(pb, qb)
                      + pearson_similarity(pb, qb)
                      + hellinger_similarity(pb, qb)) / 4)

boot_Rprob = np.array(boot_Rprob)
boot_mean = boot_Rprob.mean()
ci_low, ci_high = np.percentile(boot_Rprob, [2.5, 97.5])

# -------------------------
# Summary table
# -------------------------
summary = pd.DataFrame([
    ["JS Similarity (prob.)", JS],
    ["Bhattacharyya Coefficient", BC],
    ["Pearson Similarity (mapped to [0,1])", P_sim],
    ["Hellinger Similarity (1 − H)", H_sim],
    ["Kendall’s τ (risk ranks)", K_tau],
    ["Composite R_prob (JS, BC, Pearson, Hellinger)", R_prob],
    ["Composite R_all (+ τ)", R_all],
    ["Bootstrap R_prob mean", boot_mean],
    ["Bootstrap R_prob 95 % CI low", ci_low],
    ["Bootstrap R_prob 95 % CI high", ci_high]
], columns=["Metric", "Value"])

print(summary.to_string(index=False, float_format=lambda x: f"{x:0.6f}"))
summary.to_csv("oasis_ccna_similarity_extended.csv", index=False)

# Optional LaTeX
with open("oasis_ccna_similarity_extended.tex", "w") as f:
    f.write("\\begin{table}[ht]\\centering\\small\n")
    f.write(summary.to_latex(index=False, float_format=lambda x: f"{x:.6f}"))
    f.write("\\caption{Comprehensive similarity metrics between OASIS and CCNA models.}\\end{table}\n")

print("\n📝 Saved: oasis_ccna_similarity_extended.csv / .tex")


Bootstrapping: 100%|██████████| 1000/1000 [00:03<00:00, 330.97it/s]

                                       Metric    Value
                        JS Similarity (prob.) 0.917941
                    Bhattacharyya Coefficient 0.903654
         Pearson Similarity (mapped to [0,1]) 0.832092
                 Hellinger Similarity (1 − H) 0.689604
                     Kendall’s τ (risk ranks) 0.333333
Composite R_prob (JS, BC, Pearson, Hellinger) 0.835823
                        Composite R_all (+ τ) 0.584578
                        Bootstrap R_prob mean 0.819401
                 Bootstrap R_prob 95 % CI low 0.696377
                Bootstrap R_prob 95 % CI high 0.893745

📝 Saved: oasis_ccna_similarity_extended.csv / .tex
